# VAYU v2 — Western Ghats Climate Digital Twin
## ISRO BAH 2026 | GATv2 + Weighted CRPS + 45-day window

**Datasets required (Add Input):**
1. `shyam31415/vayu-western-ghats-processed-v1`
2. `shyam31415/vayu-ancillary-wg-v1`

**Stage 1 of 4:** `FRESH_V2`, epochs 1–25, ~7.6 hrs on T4×2 (1200 sequences × ~1100s/epoch)

**Bugs fixed vs Session 1:**
- ❌ Phase curriculum → loss-scale mismatch → no Phase 2 checkpoints saved → **REMOVED**
- ❌ Tweedie on z-scores (requires y≥0, breaks on negative normalized values) → **REPLACED with CRPS+BCE**
- ✅ Dual checkpoint: saves on val_loss improvement OR R²_rain improvement
- ✅ Checkpoint to `/kaggle/working/vayu_best.pt` every improvement (survives cancellation)

**Best model so far:** `vayu_best (3).pt` — v1 CLI run, R²_rain=0.200, R²_tmax=0.817


In [ ]:

# ── 1. GPU check + install deps ───────────────────────────────────────────────
import subprocess, sys, os
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or '⚠ No GPU!')
print('Python:', sys.version)
!pip install -q torch-geometric==2.5.3 xarray netcdf4 typer scipy
print('✓ Dependencies installed')

# ══════════════════════════════════════════════════════════════════════════════
# TRAINING MODE  ← change here before each stage
# ══════════════════════════════════════════════════════════════════════════════
TRAINING_MODE = 'FRESH_V2'   # FRESH_V2 | WARM_V2 | FRESH_V1 | WARM_V1

# Stage 1: ep 1–25 (~7.6 hrs). Stage 2: set WARM_V2 + 50. Stage 3/4: 75/100
FRESH_TOTAL_EPOCHS = 25

WARM_EXTRA_EPOCHS  = 50   # used only in WARM_V1 mode

assert TRAINING_MODE in ('WARM_V1','WARM_V2','FRESH_V2','FRESH_V1')
print(f'\nTraining mode : {TRAINING_MODE}  |  Epochs: {FRESH_TOTAL_EPOCHS}')
print(f'Session 1 target: R²_rain > 0.20 (v1 baseline)  →  0.30+ (v2 goal)')
print(f'Checkpoint saved to /kaggle/working/vayu_best.pt after every improvement')


In [ ]:

# ── 2. Mount repo + locate Kaggle datasets ────────────────────────────────────
import sys, os, shutil
from pathlib import Path

REPO_DIR = '/kaggle/working/isro'
os.makedirs(f'{REPO_DIR}/checkpoints/wg_v2', exist_ok=True)

if os.path.exists(f'{REPO_DIR}/.git'):
    os.system(f'git -C {REPO_DIR} pull --quiet')
else:
    os.system(f'rm -rf {REPO_DIR}')
    os.system(f'git clone --quiet https://github.com/Shyamistic/vayu.git {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Repo:', os.getcwd())

root = Path('/kaggle/input')

# IMD processed dataset
DATASET_DIR = None
for nc in root.rglob('normalized_2010-2025.nc'):
    DATASET_DIR = str(nc.parent); break
if not DATASET_DIR:
    raise RuntimeError("vayu-western-ghats-processed-v1 not found — add via Add Input")
print('IMD dataset:', DATASET_DIR)

# Ancillary (NCEP wind, CHIRPS, GEBCO)
ANCD_DIR = None
for nc in root.rglob('gebco_*.nc'):
    ANCD_DIR = str(nc.parent); break
print('Ancillary   :', ANCD_DIR or '⚠ NOT FOUND — add vayu-ancillary-wg-v1')

# Warm-start checkpoint (only needed for WARM_V2/WARM_V1)
V2_CKPT_PATH = None
if TRAINING_MODE in ('WARM_V1','WARM_V2'):
    for pt in sorted(root.rglob('*.pt')):
        if pt.stat().st_size > 1e6:
            V2_CKPT_PATH = str(pt); break
    print('Checkpoint  :', V2_CKPT_PATH or '⚠ NOT FOUND — will fall back to FRESH')
    if not V2_CKPT_PATH:
        TRAINING_MODE = 'FRESH_V2' if 'V2' in TRAINING_MODE else 'FRESH_V1'
        print(f'→ Falling back to {TRAINING_MODE}')


In [ ]:

# ── 3. Copy files + NCEP enrichment + build sequences ────────────────────────
import subprocess, sys, shutil, os
from pathlib import Path
import xarray as xr
import numpy as np

PY = sys.executable
PROCESSED_DIR = f'{REPO_DIR}/data/processed_western_ghats'
NCEP_DST = f'{REPO_DIR}/data/ncep_wind_subset'
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(NCEP_DST, exist_ok=True)

# Copy IMD processed files
for f in Path(DATASET_DIR).glob('*'):
    dst = Path(PROCESSED_DIR) / f.name
    if not dst.exists(): shutil.copy2(f, dst)
print(f'✓ IMD processed files copied')

# Copy ancillary (NCEP wind, CHIRPS, GEBCO)
if ANCD_DIR:
    n_ncep = 0
    for pat in ['uwnd_*.nc','vwnd_*.nc','shum_*.nc','pr_wtr_*.nc']:
        for f in Path(ANCD_DIR).glob(pat):
            shutil.copy2(f, NCEP_DST); n_ncep += 1
    for f in Path(ANCD_DIR).glob('chirps_*.nc'):
        dst_d = Path(REPO_DIR)/'chirps'; dst_d.mkdir(exist_ok=True)
        shutil.copy2(f, dst_d)
    for f in Path(ANCD_DIR).glob('gebco_*.nc'):
        dst_d = Path(REPO_DIR)/'gebco'; dst_d.mkdir(exist_ok=True)
        shutil.copy2(f, dst_d)
    print(f'✓ Ancillary copied: {n_ncep} NCEP files | CHIRPS | GEBCO')

# NCEP enrichment — add uwnd_850, vwnd_850, shum_850 to normalized file
NORM_FILE = f'{PROCESSED_DIR}/normalized_2010-2025.nc'
ds_check = xr.open_dataset(NORM_FILE)
existing = list(ds_check.data_vars); ds_check.close()
needs_ncep = not any('uwnd' in v for v in existing)
print(f'\nNormalized vars: {existing}')
print(f'NCEP enrichment needed: {needs_ncep}')

if needs_ncep and any(Path(NCEP_DST).glob('uwnd_*.nc')):
    print('Enriching with NCEP 850 hPa wind + humidity...')
    ds = xr.open_dataset(NORM_FILE)
    lats, lons, times = ds.lat.values, ds.lon.values, ds.time.values
    added = []
    for varname, pattern in [('uwnd_850','uwnd_*.nc'),('vwnd_850','vwnd_*.nc'),('shum_850','shum_*.nc')]:
        files = sorted(Path(NCEP_DST).glob(pattern))
        if not files or varname in ds.data_vars: continue
        try:
            parts = []
            for f in files:
                d = xr.open_dataset(f); rv = list(d.data_vars)[0]; arr = d[rv]
                rmap = {dim:'lat' for dim in arr.dims if dim.lower() in ('latitude','nav_lat')}
                rmap.update({dim:'lon' for dim in arr.dims if dim.lower() in ('longitude','nav_lon')})
                if rmap: arr = arr.rename(rmap)
                parts.append(arr); d.close()
            comb = xr.concat(parts, dim='time', join='override').sortby('time')
            rg = comb.interp({'lat':lats,'lon':lons}, method='linear')
            rg = rg.interp(time=times, method='nearest').fillna(0.0)
            mean_v, std_v = float(rg.values.mean()), max(float(rg.values.std()), 1e-6)
            ds[varname] = (rg - mean_v) / std_v
            added.append(varname)
            print(f'  ✓ {varname}: {len(files)} years | mean={mean_v:.3f} std={std_v:.3f}')
        except Exception as e:
            print(f'  ⚠ {varname}: {e}')
    if added:
        tmp = NORM_FILE + '.tmp'; ds.to_netcdf(tmp); ds.close(); shutil.move(tmp, NORM_FILE)
        ds2 = xr.open_dataset(NORM_FILE)
        print(f'✓ Enriched vars: {list(ds2.data_vars)}'); ds2.close()
    else:
        ds.close()

# Build sequences: 1200 train / 200 val (reduces epoch time ~1100s vs 1466s)
# 25 epochs × 1100s ≈ 7.6 hrs — fits within 12hr Kaggle session
print('\n=== Building sequences (45d window, 1200 train / 200 val) ===')
r = subprocess.run(
    [PY, '-m', 'data_ingestion.cli', 'build-sequences',
     '--normalized-file', NORM_FILE, '--input-window', '45',
     '--target-window', '7', '--max-train', '1200',
     '--max-val', '200', '--stride', '3', '--output-dir', PROCESSED_DIR],
    capture_output=True, text=True, cwd=REPO_DIR
)
if r.returncode == 0:
    print(r.stdout[-500:]); print('✓ Sequences built')
else:
    print('⚠ build-sequences failed'); print(r.stderr[-400:])

for f in Path(PROCESSED_DIR).glob('*.pt'):
    print(f'  {f.name}: {f.stat().st_size/1e6:.0f} MB')


In [ ]:

# ── 4. Model architecture: VayuClimateModelV2 ─────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv

class PhysicsFeatureEncoder(nn.Module):
    def __init__(self, in_channels: int, hidden: int):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_channels, hidden), nn.LayerNorm(hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.SiLU(),
        )
        self.orographic_gate = nn.Sequential(nn.Linear(hidden, hidden), nn.Sigmoid())
    def forward(self, x):
        h = self.proj(x)
        return h * self.orographic_gate(h)

class GATv2Block(nn.Module):
    def __init__(self, hidden: int, heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden % heads == 0
        self.gat = GATv2Conv(hidden, hidden // heads, heads=heads, dropout=dropout, add_self_loops=True)
        self.norm = nn.LayerNorm(hidden)
        self.ff = nn.Sequential(nn.Linear(hidden, hidden * 2), nn.GELU(), nn.Linear(hidden * 2, hidden))
        self.norm2 = nn.LayerNorm(hidden)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, edge_index):
        h = self.norm(x + self.drop(self.gat(x, edge_index)))
        return self.norm2(h + self.drop(self.ff(h)))

class TwoStageRainfallHead(nn.Module):
    """Stage 1: binary occurrence BCE.  Stage 2: log-normal amount via CRPS."""
    def __init__(self, d_model: int, target_steps: int):
        super().__init__()
        self.target_steps = target_steps
        self.occurrence = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, target_steps))
        self.amount    = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, target_steps))
        self.sigma     = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, target_steps), nn.Softplus())
    def forward(self, z):
        occ   = self.occurrence(z)          # logits
        mu    = self.amount(z)              # log-space mean
        sigma = self.sigma(z).clamp(0.01, 5.0)
        return occ, mu, sigma

class VayuClimateModelV2(nn.Module):
    def __init__(self, in_channels: int = 6, gnn_hidden: int = 192, d_model: int = 384,
                 n_heads: int = 4, n_gat_layers: int = 4, n_transformer_layers: int = 6,
                 target_steps: int = 7, seq_len: int = 45, n_nodes: int = 0,
                 dropout: float = 0.1):
        super().__init__()
        self.gnn_hidden   = gnn_hidden
        self.d_model      = d_model
        self.n_nodes      = n_nodes
        self.target_steps = target_steps

        self.encoder = nn.Sequential(nn.Linear(in_channels, gnn_hidden), nn.LayerNorm(gnn_hidden), nn.SiLU())
        self.gat_layers = nn.ModuleList([GATv2Block(gnn_hidden, n_heads, dropout) for _ in range(n_gat_layers)])
        self.temporal_proj = nn.Linear(gnn_hidden * seq_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
                                                   dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_transformer_layers)
        self.rain_head  = TwoStageRainfallHead(d_model, target_steps)
        self.tmax_head  = nn.Sequential(nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, target_steps))
        self.tmin_head  = nn.Sequential(nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, target_steps))

    def forward(self, x, edge_index, batch_map=None):
        B, T, N, F = x.shape
        h = self.encoder(x.view(B*T*N, F)).view(B, T, N, self.gnn_hidden)
        gat_out = []
        for t in range(T):
            xt = h[:, t, :, :].view(B*N, self.gnn_hidden)
            for gat in self.gat_layers: xt = gat(xt, edge_index)
            gat_out.append(xt.view(B, N, self.gnn_hidden))
        gat_seq = torch.stack(gat_out, dim=1)                   # (B, T, N, H)
        node_seq = gat_seq.permute(0, 2, 1, 3).contiguous().view(B*N, T*self.gnn_hidden)
        node_emb = F.relu(self.temporal_proj(node_seq)).view(B, N, self.d_model)
        ctx = self.transformer(node_emb)                         # (B, N, d_model)
        occ, mu, sigma = self.rain_head(ctx)
        tmax = self.tmax_head(ctx)
        tmin = self.tmin_head(ctx)
        return occ, mu, sigma, tmax, tmin

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Quick sanity forward pass
_m = VayuClimateModelV2(in_channels=6, gnn_hidden=192, d_model=384).to(DEVICE)
_x = torch.randn(2, 45, 10, 6, device=DEVICE)
_ei = torch.zeros(2, 0, dtype=torch.long, device=DEVICE)
with torch.no_grad():
    o, mu, s, tmax, tmin = _m(_x, _ei)
print(f'Forward pass OK: occ={tuple(o.shape)} mu={tuple(mu.shape)} sigma={tuple(s.shape)} tmax={tuple(tmax.shape)}')
params = sum(p.numel() for p in _m.parameters())
print(f'Parameters: {params/1e6:.2f}M')
del _m, _x, _ei


In [ ]:

# ── 5. VayuV2Loss — weighted CRPS + BCE (NO Tweedie) ─────────────────────────
# Tweedie requires y≥0 but z-score normalized rainfall goes negative on dry days
# → val_loss spikes to 202, R²_rain = -0.35.  Fixed: CRPS + BCE only.

import torch
import torch.nn as nn
import torch.nn.functional as F

class WeightedCRPSLoss(nn.Module):
    """CRPS for probabilistic amount prediction (log-normal parametrization)."""
    def __init__(self, alpha: float = 3.0, heavy_threshold: float = 20.0):
        super().__init__()
        self.alpha = alpha
        self.heavy_threshold = heavy_threshold  # mm/day (in normalized space ~1.5σ)
    def forward(self, mu, sigma, target):
        """mu, sigma: (B,N,T)  target: (B,N,T) z-scored rainfall"""
        sigma = sigma.clamp(min=1e-6)
        z = (target - mu) / sigma
        normal = torch.distributions.Normal(0, 1)
        phi_z  = normal.log_prob(z).exp()
        Phi_z  = normal.cdf(z)
        crps   = sigma * (z * (2*Phi_z - 1) + 2*phi_z - 1/torch.pi**0.5)
        heavy_mask = (target > self.heavy_threshold / 30.0).float()   # rough z threshold
        weights = 1.0 + (self.alpha - 1.0) * heavy_mask
        return (crps * weights).mean()

class VayuV2Loss(nn.Module):
    """
    VAYU v2 loss — scientifically correct for zero-inflated precipitation.
    = 70% weighted CRPS (amount, log-normal)
    + 30% BCE       (occurrence, binary rain/no-rain)
    """
    def __init__(self):
        super().__init__()
        self.crps  = WeightedCRPSLoss(alpha=3.0)
        self.rain_w = 1.8  # down-weight balanced with tmax/tmin

    def forward(self, occ, mu, sigma, tmax_pred, tmin_pred,
                rain_true, tmax_true, tmin_true):
        # Occurrence: binary BCE — 0 = dry (< 1mm raw)
        DRY_THRESH = -0.3       # approximate z-score for 1mm/day
        rain_bin   = (rain_true > DRY_THRESH).float()
        bce        = F.binary_cross_entropy_with_logits(occ, rain_bin)
        # Amount: CRPS on log-normal distribution
        crps_loss  = self.crps(mu, sigma, rain_true)
        # Deterministic heads: MSE
        tmax_loss  = F.mse_loss(tmax_pred, tmax_true)
        tmin_loss  = F.mse_loss(tmin_pred, tmin_true)
        # Combined
        rain_total = 0.7 * crps_loss + 0.3 * bce
        total = (self.rain_w * rain_total + tmax_loss + tmin_loss) / (self.rain_w + 2.0)
        return total, {'crps': crps_loss.item(), 'bce': bce.item(),
                       'tmax': tmax_loss.item(), 'tmin': tmin_loss.item()}

# Quick test
criterion = VayuV2Loss()
_occ  = torch.randn(2, 10, 7)
_mu   = torch.randn(2, 10, 7)
_sig  = torch.ones(2, 10, 7)
_rain = torch.randn(2, 10, 7)
_loss, _comps = criterion(_occ, _mu, _sig, _mu, _mu, _rain, _rain, _rain)
print(f'VayuV2Loss sanity: {_loss.item():.4f}  comps={_comps}')
assert _loss.item() < 100, 'Loss too large — check CRPS/BCE'
print('✓ Loss functions OK')


In [ ]:

# ── 6. Checkpoint warm-start loader ──────────────────────────────────────────
def load_checkpoint_v2(ckpt_path: str, device):
    """Load any .pt checkpoint — auto-detects v1 vs v2 architecture."""
    sd = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    if isinstance(sd, dict) and 'model_state_dict' in sd:
        state_dict = sd['model_state_dict']
        meta       = sd.get('metadata', {})
    else:
        state_dict, meta = sd, {}

    # Auto-detect architecture from weight shapes
    enc_w = state_dict.get('encoder.input_proj.0.weight') or state_dict.get('encoder.0.weight')
    is_v2 = enc_w is None   # v2 encoder is named differently

    print(f'Checkpoint: {ckpt_path}')
    print(f'  Epoch:    {meta.get("epoch", "?")}')
    print(f'  Arch:     {"v2" if is_v2 else "v1"} (auto-detected)')
    print(f'  R²_rain:  {meta.get("r2_rain", "?")}')
    return state_dict, meta

CHECKPOINT_PATH = None
START_EPOCH     = 0

if TRAINING_MODE in ('WARM_V1','WARM_V2') and V2_CKPT_PATH:
    state_dict, meta = load_checkpoint_v2(V2_CKPT_PATH, DEVICE)
    START_EPOCH = meta.get('epoch', 0)
    TOTAL_EPOCHS = START_EPOCH + WARM_EXTRA_EPOCHS
    CHECKPOINT_PATH = V2_CKPT_PATH
    print(f'\n→ Warm start from epoch {START_EPOCH}, training to {TOTAL_EPOCHS}')
elif TRAINING_MODE in ('FRESH_V2','FRESH_V1'):
    TOTAL_EPOCHS = FRESH_TOTAL_EPOCHS
    print(f'→ Fresh start, training epochs 1–{TOTAL_EPOCHS}')
else:
    TOTAL_EPOCHS = FRESH_TOTAL_EPOCHS

print(f'Training mode: {TRAINING_MODE}  |  Epochs to train: {TOTAL_EPOCHS - START_EPOCH}')


In [ ]:

# ── 7. Data loading ───────────────────────────────────────────────────────────
import torch
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.data import Data
from pathlib import Path
import numpy as np

PROCESSED_DIR = f'{REPO_DIR}/data/processed_western_ghats'

def load_sequences(processed_dir: str):
    """Load pre-built sequence tensors + graph topology."""
    d = Path(processed_dir)
    X_tr  = torch.load(d / 'X_train.pt', weights_only=False)
    y_tr  = torch.load(d / 'y_train.pt', weights_only=False)
    X_val = torch.load(d / 'X_val.pt',   weights_only=False)
    y_val = torch.load(d / 'y_val.pt',   weights_only=False)
    graph = torch.load(d / 'graph.pt',   weights_only=False)
    ei = graph.edge_index if hasattr(graph, 'edge_index') else graph.get('edge_index', torch.zeros(2,0,dtype=torch.long))
    print(f'Train: {X_tr.shape}  Val: {X_val.shape}')
    print(f'Nodes: {X_tr.shape[2]}  Features: {X_tr.shape[3]}  Edge pairs: {ei.shape[1]}')
    return X_tr, y_tr, X_val, y_val, ei

def unpack_seq(y_batch):
    """y_batch: (B, N, 3*T) → rain (B,N,T), tmax (B,N,T), tmin (B,N,T)"""
    T = y_batch.shape[-1] // 3
    return y_batch[:,:,:T], y_batch[:,:,T:2*T], y_batch[:,:,2*T:]

X_tr, y_tr, X_val, y_val, EDGE_INDEX = load_sequences(PROCESSED_DIR)
EDGE_INDEX = EDGE_INDEX.to(DEVICE)

N_NODES   = X_tr.shape[2]
IN_FEAT   = X_tr.shape[3]
SEQ_LEN   = X_tr.shape[1]
TARGET_T  = y_tr.shape[-1] // 3
BATCH_SIZE = 4

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True,  pin_memory=True, drop_last=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
print(f'Batches — train: {len(train_loader)}  val: {len(val_loader)}')
print(f'Config  — N_NODES:{N_NODES} IN_FEAT:{IN_FEAT} SEQ_LEN:{SEQ_LEN} TARGET_T:{TARGET_T}')


In [ ]:

# ── 8. Training loop ──────────────────────────────────────────────────────────
# NO phase curriculum (removed — caused checkpoint loss-scale mismatch)
# Full VayuV2Loss from epoch 1
# Dual checkpoint: saves if val_loss improves OR if R²_rain improves by >0.005

import math, time

# ── Build model ────────────────────────────────────────────────────────────────
model = VayuClimateModelV2(
    in_channels=IN_FEAT, gnn_hidden=192, d_model=384,
    n_heads=4, n_gat_layers=4, n_transformer_layers=6,
    target_steps=TARGET_T, seq_len=SEQ_LEN, n_nodes=N_NODES,
).to(DEVICE)

if CHECKPOINT_PATH and TRAINING_MODE in ('WARM_V1','WARM_V2'):
    sd, _ = load_checkpoint_v2(CHECKPOINT_PATH, DEVICE)
    model.load_state_dict(sd, strict=False)
    print(f'✓ Warm-start weights loaded from {CHECKPOINT_PATH}')

criterion = VayuV2Loss()
optimizer  = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS, eta_min=1e-6)

# Advance LR scheduler for warm-start
for _ in range(START_EPOCH): scheduler.step()

CKPT_DIR = f'{REPO_DIR}/checkpoints/wg_v2'
CKPT_ROOT = '/kaggle/working/vayu_best.pt'   # always readable even after time limit
best_val_loss = float('inf')
best_r2_rain  = -float('inf')

print(f'\n{"─"*70}')
print(f'Training VayuClimateModelV2  |  {sum(p.numel() for p in model.parameters())/1e6:.2f}M params')
print(f'Epochs {START_EPOCH+1}–{TOTAL_EPOCHS}  |  Batch {BATCH_SIZE}  |  LR {optimizer.param_groups[0]["lr"]:.1e}')
print(f'{"─"*70}')

def r2_score(pred, true):
    ss_res = ((pred - true)**2).sum()
    ss_tot = ((true - true.mean())**2).sum()
    return float(1 - ss_res / (ss_tot + 1e-8))

scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')

for epoch in range(START_EPOCH + 1, TOTAL_EPOCHS + 1):
    t0 = time.time()

    # ── Train ──────────────────────────────────────────────────────────────────
    model.train()
    tr_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        rain_t, tmax_t, tmin_t = unpack_seq(yb)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
            occ, mu, sigma, tmax_p, tmin_p = model(xb, EDGE_INDEX)
            loss, comps = criterion(occ, mu, sigma, tmax_p, tmin_p, rain_t, tmax_t, tmin_t)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        tr_loss += loss.item()
    tr_loss /= len(train_loader)

    # ── Validation ─────────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    rain_preds, rain_true_all = [], []
    tmax_preds, tmax_true_all = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            rain_t, tmax_t, tmin_t = unpack_seq(yb)
            with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
                occ, mu, sigma, tmax_p, tmin_p = model(xb, EDGE_INDEX)
                loss, _ = criterion(occ, mu, sigma, tmax_p, tmin_p, rain_t, tmax_t, tmin_t)
            val_loss += loss.item()
            rain_preds.append(mu.cpu()); rain_true_all.append(rain_t.cpu())
            tmax_preds.append(tmax_p.cpu()); tmax_true_all.append(tmax_t.cpu())
    val_loss /= len(val_loader)

    rain_pred_all = torch.cat(rain_preds)
    rain_true_cat = torch.cat(rain_true_all)
    tmax_pred_all = torch.cat(tmax_preds)
    tmax_true_cat = torch.cat(tmax_true_all)
    r2_rain = r2_score(rain_pred_all, rain_true_cat)
    r2_tmax = r2_score(tmax_pred_all, tmax_true_cat)

    scheduler.step()
    elapsed = time.time() - t0

    print(f'Ep {epoch:3d}/{TOTAL_EPOCHS} | train={tr_loss:.4f} val={val_loss:.4f} '
          f'| R²_rain={r2_rain:.3f} R²_tmax={r2_tmax:.3f} | {elapsed:.0f}s | lr={scheduler.get_last_lr()[0]:.1e}')

    # ── Checkpoint (dual metric) ────────────────────────────────────────────────
    save_ckpt = False
    reason = []
    if val_loss < best_val_loss - 1e-5:
        best_val_loss = val_loss; save_ckpt = True; reason.append(f'val_loss↓{val_loss:.4f}')
    if r2_rain > best_r2_rain + 0.005:
        best_r2_rain = r2_rain; save_ckpt = True; reason.append(f'R²_rain↑{r2_rain:.3f}')

    if save_ckpt:
        meta = {'epoch': epoch, 'val_loss': val_loss, 'r2_rain': r2_rain, 'r2_tmax': r2_tmax,
                'training_mode': TRAINING_MODE, 'best_val_loss': best_val_loss, 'best_r2_rain': best_r2_rain}
        payload = {'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'metadata': meta}
        torch.save(payload, CKPT_ROOT)
        torch.save(payload, f'{CKPT_DIR}/vayu_v2_ep{epoch:03d}.pt')
        print(f'    ✓ Saved checkpoint ({" | ".join(reason)})')

print(f'\n✓ Training complete  |  best_val_loss={best_val_loss:.4f}  best_R²_rain={best_r2_rain:.3f}')


In [ ]:

# ── 9. Evaluation + visualisation ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

print('=== Final Evaluation ===')
model.eval()
all_rain_p, all_rain_t = [], []
all_tmax_p, all_tmax_t = [], []
all_tmin_p, all_tmin_t = [], []

with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        rain_t, tmax_t, tmin_t = unpack_seq(yb)
        occ, mu, sigma, tmax_p, tmin_p = model(xb, EDGE_INDEX)
        all_rain_p.append(mu.cpu()); all_rain_t.append(rain_t.cpu())
        all_tmax_p.append(tmax_p.cpu()); all_tmax_t.append(tmax_t.cpu())
        all_tmin_p.append(tmin_p.cpu()); all_tmin_t.append(tmin_t.cpu())

rp = torch.cat(all_rain_p).flatten().numpy()
rt = torch.cat(all_rain_t).flatten().numpy()
tp = torch.cat(all_tmax_p).flatten().numpy()
tt = torch.cat(all_tmax_t).flatten().numpy()

def r2(p, t):
    return 1 - ((p-t)**2).sum() / (((t-t.mean())**2).sum() + 1e-8)

r2_rain = r2(rp, rt)
r2_tmax = r2(tp, tt)
print(f'R²_rain : {r2_rain:.4f}  (v1 baseline 0.200  →  v2 target 0.300)')
print(f'R²_tmax : {r2_tmax:.4f}  (v1 baseline 0.817)')

fig, axs = plt.subplots(1, 3, figsize=(15, 5))
axs[0].scatter(rt[:2000], rp[:2000], alpha=0.3, s=2, color='steelblue')
axs[0].set_xlabel('True rainfall (z-score)'); axs[0].set_ylabel('Predicted'); axs[0].set_title(f'Rainfall R²={r2_rain:.3f}')
axs[0].axline((0,0), slope=1, color='red', lw=1)

axs[1].scatter(tt[:2000], tp[:2000], alpha=0.3, s=2, color='orange')
axs[1].set_xlabel('True Tmax (z-score)'); axs[1].set_title(f'Tmax R²={r2_tmax:.3f}')
axs[1].axline((0,0), slope=1, color='red', lw=1)

epochs_done = TOTAL_EPOCHS - START_EPOCH
axs[2].text(0.5, 0.5, f'Epochs: {START_EPOCH+1}–{TOTAL_EPOCHS}\n'
            f'Best val_loss: {best_val_loss:.4f}\nBest R²_rain: {best_r2_rain:.3f}\nBest R²_tmax: {r2_tmax:.3f}',
            ha='center', va='center', transform=axs[2].transAxes, fontsize=14)
axs[2].axis('off'); axs[2].set_title('Summary')
plt.tight_layout()
plt.savefig('/kaggle/working/eval_plots.png', dpi=150)
plt.show()
print('✓ Plots saved to /kaggle/working/eval_plots.png')


In [ ]:

# ── 10. Package checkpoint for download ───────────────────────────────────────
import os, shutil, time

best_ckpt = '/kaggle/working/vayu_best.pt'
if os.path.exists(best_ckpt):
    info = torch.load(best_ckpt, map_location='cpu', weights_only=False)
    meta = info.get('metadata', {})
    print('=== Best Checkpoint ===')
    print(f'  Epoch     : {meta.get("epoch","?")}')
    print(f'  val_loss  : {meta.get("val_loss","?"):.4f}')
    print(f'  R²_rain   : {meta.get("r2_rain","?"):.4f}')
    print(f'  R²_tmax   : {meta.get("r2_tmax","?"):.4f}')
    print(f'  File size : {os.path.getsize(best_ckpt)/1e6:.1f} MB')
    print(f'\n→ Download vayu_best.pt from Kaggle Output → Data → /kaggle/working/')
    print(f'\nNext step: upload to Kaggle dataset shyam31415/vayu-v2-checkpoint')
    print(f'           then run Stage 2 with TRAINING_MODE = "WARM_V2"')
else:
    print('⚠ No checkpoint found at /kaggle/working/vayu_best.pt')
    print('  Check that training ran successfully and saved at least one checkpoint.')
